In [3]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data/cleaned/brain-tumor-dataset")
OUTPUT_DIR = Path("../data/splits")
RANDOM_SEED = 42

LABEL_REMAP = {
    "glioma": "glioma",
    "meningioma": "meningioma",
    "no_tumor": "notumor",
    "pituitary": "pituitary",
}

for split in ["train", "val", "test"]:
    for cls in LABEL_REMAP.values():
        (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

for cls_folder in sorted(DATA_DIR.iterdir()):
    if not cls_folder.is_dir():
        continue

    label = LABEL_REMAP.get(cls_folder.name)
    if label is None:
        continue

    images = sorted([
        p for p in cls_folder.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

    train, temp = train_test_split(images, test_size=0.20, random_state=RANDOM_SEED)
    val, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_SEED)

    for split_name, split_images in [("train", train), ("val", val), ("test", test)]:
        for img in split_images:
            shutil.copy2(img, OUTPUT_DIR / split_name / label / img.name)

    print(f"{cls_folder.name}: train={len(train)} val={len(val)} test={len(test)}")

print("\nDone! Structure:")
for split in ["train", "val", "test"]:
    total = sum(
        len(list((OUTPUT_DIR / split / c).iterdir()))
        for c in LABEL_REMAP.values()
    )
    print(f"  data/splits/{split}/ → {total:,} images")

glioma: train=2990 val=374 test=374
meningioma: train=1791 val=224 test=224
no_tumor: train=1385 val=173 test=174
pituitary: train=2155 val=269 test=270

Done! Structure:
  data/splits/train/ → 8,321 images
  data/splits/val/ → 1,040 images
  data/splits/test/ → 1,042 images
